# Food Demand Prediction Using Machine Learning

This notebook contains the ML research and experimentation for predicting food demand in restaurants, cafés, canteens, and food service operations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import shap
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Generate Sample Dataset

Create a realistic food demand dataset with historical sales data.

In [ ]:
# Generate sample dataset
np.random.seed(42)

# Date range: 2 years of data
date_range = pd.date_range(start='2024-01-01', end='2025-12-31', freq='D')

# Food items
food_items = ['Pizza', 'Burger', 'Pasta', 'Salad', 'Sandwich', 'Rice Bowl', 'Noodles', 'Soup']

# Create dataset
data = []
for date in date_range:
    for food in food_items:
        # Base demand varies by food item
        base_demand = {'Pizza': 80, 'Burger': 100, 'Pasta': 70, 'Salad': 50, 
                      'Sandwich': 90, 'Rice Bowl': 85, 'Noodles': 75, 'Soup': 40}[food]
        
        # Weekend boost
        weekend_boost = 1.3 if date.dayofweek >= 5 else 1.0
        
        # Holiday boost (simulate holidays)
        is_holiday = 1 if (date.month == 12 and date.day >= 20) or \
                         (date.month == 1 and date.day <= 5) or \
                         (date.month == 7 and date.day <= 7) else 0
        holiday_boost = 1.5 if is_holiday else 1.0
        
        # Day of week effect
        day_effect = [0.9, 0.95, 1.0, 1.05, 1.15, 1.3, 1.2][date.dayofweek]
        
        # Calculate demand with some randomness
        demand = int(base_demand * weekend_boost * holiday_boost * day_effect * 
                    np.random.uniform(0.8, 1.2))
        
        data.append({
            'date': date,
            'food_item': food,
            'demand': demand,
            'holiday': is_holiday
        })

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head(10)

## 2. Exploratory Data Analysis

In [ ]:
# Basic statistics
print("Dataset Info:")
print(df.info())
print("\nBasic Statistics:")
print(df.describe())
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Demand distribution by food item
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Average demand by food item
df.groupby('food_item')['demand'].mean().sort_values().plot(kind='barh', ax=axes[0,0], color='skyblue')
axes[0,0].set_title('Average Demand by Food Item')
axes[0,0].set_xlabel('Average Demand')

# Demand over time
df.groupby('date')['demand'].sum().plot(ax=axes[0,1], color='green')
axes[0,1].set_title('Total Daily Demand Over Time')
axes[0,1].set_ylabel('Total Demand')

# Weekend vs Weekday
df['is_weekend'] = df['date'].dt.dayofweek >= 5
df.groupby('is_weekend')['demand'].mean().plot(kind='bar', ax=axes[1,0], color=['coral', 'lightgreen'])
axes[1,0].set_title('Average Demand: Weekday vs Weekend')
axes[1,0].set_xticklabels(['Weekday', 'Weekend'], rotation=0)
axes[1,0].set_ylabel('Average Demand')

# Holiday vs Non-Holiday
df.groupby('holiday')['demand'].mean().plot(kind='bar', ax=axes[1,1], color=['steelblue', 'orange'])
axes[1,1].set_title('Average Demand: Non-Holiday vs Holiday')
axes[1,1].set_xticklabels(['Non-Holiday', 'Holiday'], rotation=0)
axes[1,1].set_ylabel('Average Demand')

plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
# Create time-based features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['quarter'] = df['date'].dt.quarter
df['week_of_year'] = df['date'].dt.isocalendar().week

# Encode food items
le_food = LabelEncoder()
df['food_item_encoded'] = le_food.fit_transform(df['food_item'])

# Calculate rolling statistics (lag features)
df = df.sort_values(['food_item', 'date'])
df['demand_lag_7'] = df.groupby('food_item')['demand'].shift(7)
df['demand_lag_30'] = df.groupby('food_item')['demand'].shift(30)
df['demand_rolling_mean_7'] = df.groupby('food_item')['demand'].rolling(window=7, min_periods=1).mean().reset_index(0, drop=True)
df['demand_rolling_mean_30'] = df.groupby('food_item')['demand'].rolling(window=30, min_periods=1).mean().reset_index(0, drop=True)

# Fill NaN values from lag features
df['demand_lag_7'] = df['demand_lag_7'].fillna(df['demand_rolling_mean_7'])
df['demand_lag_30'] = df['demand_lag_30'].fillna(df['demand_rolling_mean_30'])

print("Features created:")
print(df.columns.tolist())
print(f"\nDataset shape after feature engineering: {df.shape}")
df.head()

## 4. Prepare Data for Modeling

In [ ]:
# Select features for modeling
feature_columns = ['food_item_encoded', 'year', 'month', 'day', 'day_of_week', 
                  'is_weekend', 'quarter', 'week_of_year', 'holiday',
                  'demand_lag_7', 'demand_lag_30', 'demand_rolling_mean_7', 'demand_rolling_mean_30']

X = df[feature_columns]
y = df['demand']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## 5. Model Training and Evaluation

### 5.1 Linear Regression

In [ ]:
# Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test)

# Evaluation
mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression Results:")
print(f"MAE: {mae_lr:.2f}")
print(f"MSE: {mse_lr:.2f}")
print(f"RMSE: {rmse_lr:.2f}")
print(f"R² Score: {r2_lr:.4f}")

### 5.2 Random Forest

In [ ]:
# Random Forest
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)

# Evaluation
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Results:")
print(f"MAE: {mae_rf:.2f}")
print(f"MSE: {mse_rf:.2f}")
print(f"RMSE: {rmse_rf:.2f}")
print(f"R² Score: {r2_rf:.4f}")

### 5.3 Deep Learning (Neural Network)

In [ ]:
# Build neural network
dl_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

dl_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train
history = dl_model.fit(X_train_scaled, y_train, 
                      validation_split=0.2, 
                      epochs=50, 
                      batch_size=32, 
                      verbose=0)

# Predictions
y_pred_dl = dl_model.predict(X_test_scaled, verbose=0).flatten()

# Evaluation
mae_dl = mean_absolute_error(y_test, y_pred_dl)
mse_dl = mean_squared_error(y_test, y_pred_dl)
rmse_dl = np.sqrt(mse_dl)
r2_dl = r2_score(y_test, y_pred_dl)

print("Deep Learning Results:")
print(f"MAE: {mae_dl:.2f}")
print(f"MSE: {mse_dl:.2f}")
print(f"RMSE: {rmse_dl:.2f}")
print(f"R² Score: {r2_dl:.4f}")

## 6. Model Comparison

In [ ]:
# Create comparison table
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'Deep Learning'],
    'MAE': [mae_lr, mae_rf, mae_dl],
    'MSE': [mse_lr, mse_rf, mse_dl],
    'RMSE': [rmse_lr, rmse_rf, rmse_dl],
    'R²': [r2_lr, r2_rf, r2_dl]
})

print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)
print(results.to_string(index=False))
print("="*70)

# Identify best model
best_model_idx = results['MAE'].idxmin()
best_model_name = results.loc[best_model_idx, 'Model']
print(f"\n🏆 Best Model: {best_model_name} (Lowest MAE)")

## 7. Explainable AI (SHAP)

Use SHAP to explain Random Forest predictions.

In [ ]:
# Create SHAP explainer for Random Forest
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test.iloc[:100])  # Sample for visualization

# Summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test.iloc[:100], feature_names=feature_columns, show=False)
plt.title('SHAP Feature Importance Summary')
plt.tight_layout()
plt.show()

print("SHAP values calculated successfully!")
print("Features ranked by importance:")
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': np.abs(shap_values).mean(axis=0)
}).sort_values('importance', ascending=False)
print(feature_importance)

## 8. Save Models and Artifacts

In [ ]:
import joblib
import os

# Create models directory
os.makedirs('../models', exist_ok=True)
os.makedirs('../data', exist_ok=True)

# Save models
joblib.dump(lr_model, '../models/linear_regression_model.pkl')
joblib.dump(rf_model, '../models/random_forest_model.pkl')
dl_model.save('../models/deep_learning_model.h5')

# Save preprocessors
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(le_food, '../models/label_encoder.pkl')

# Save feature columns
joblib.dump(feature_columns, '../models/feature_columns.pkl')

# Save sample dataset
df.to_csv('../data/food_demand_data.csv', index=False)

# Save model comparison results
results.to_csv('../models/model_comparison.csv', index=False)

print("✅ All models and artifacts saved successfully!")
print("\nSaved files:")
print("- linear_regression_model.pkl")
print("- random_forest_model.pkl")
print("- deep_learning_model.h5")
print("- scaler.pkl")
print("- label_encoder.pkl")
print("- feature_columns.pkl")
print("- food_demand_data.csv")
print("- model_comparison.csv")

## 9. Example Prediction

In [ ]:
# Example: Predict demand for Pizza on a specific date
def predict_demand(food_item, date, holiday, model='rf'):
    """
    Predict demand for a food item on a specific date.
    
    Parameters:
    - food_item: str, name of food item
    - date: datetime, prediction date
    - holiday: int, 1 if holiday, 0 otherwise
    - model: str, 'lr', 'rf', or 'dl'
    """
    # Get historical data for the food item
    food_data = df[df['food_item'] == food_item].copy()
    
    if len(food_data) == 0:
        return None, "Food item not found"
    
    # Create features for prediction
    pred_date = pd.to_datetime(date)
    
    features = {
        'food_item_encoded': le_food.transform([food_item])[0],
        'year': pred_date.year,
        'month': pred_date.month,
        'day': pred_date.day,
        'day_of_week': pred_date.dayofweek,
        'is_weekend': 1 if pred_date.dayofweek >= 5 else 0,
        'quarter': pred_date.quarter,
        'week_of_year': pred_date.isocalendar()[1],
        'holiday': holiday,
        'demand_lag_7': food_data['demand'].iloc[-7:].mean(),
        'demand_lag_30': food_data['demand'].iloc[-30:].mean(),
        'demand_rolling_mean_7': food_data['demand'].iloc[-7:].mean(),
        'demand_rolling_mean_30': food_data['demand'].iloc[-30:].mean()
    }
    
    X_pred = pd.DataFrame([features])[feature_columns]
    
    # Make prediction
    if model == 'lr':
        prediction = lr_model.predict(X_pred)[0]
    elif model == 'rf':
        prediction = rf_model.predict(X_pred)[0]
    elif model == 'dl':
        X_pred_scaled = scaler.transform(X_pred)
        prediction = dl_model.predict(X_pred_scaled, verbose=0)[0][0]
    
    return int(prediction), features

# Example prediction
test_food = 'Pizza'
test_date = '2026-09-05'  # Saturday
test_holiday = 0

prediction, features = predict_demand(test_food, test_date, test_holiday, model='rf')

print(f"\n🔮 Prediction Example:")
print(f"Food Item: {test_food}")
print(f"Date: {test_date}")
print(f"Holiday: {'Yes' if test_holiday else 'No'}")
print(f"\n📊 Predicted Demand: {prediction} orders")
print(f"\n💡 Recommendation: Prepare approximately {prediction} servings of {test_food}")

## Conclusion

This notebook demonstrates:
1. ✅ Data generation and exploration
2. ✅ Feature engineering with time-based and lag features
3. ✅ Three ML models: Linear Regression, Random Forest, Deep Learning
4. ✅ Model comparison and evaluation
5. ✅ Explainable AI using SHAP
6. ✅ Model persistence and artifact saving
7. ✅ Example prediction workflow

The trained models and artifacts are ready to be integrated into a production web application.